# Forest generation showcase
A step-by-step, reproducible walkthrough that visualises each step of our forest generation algorithm.


## 0. Scene parameters
Define all configurable parameters for the preview so the rest of the notebook can reference them.


In [ ]:
terrain_params = {
    'size': 50,
    'resolution': 0.25,
    'scale': 4.0,
    'octaves': 2,
    'height_scale': 2,
    'apply_microrelief': True,
    'moisture_weights': {'flow': 0.55, 'slope': 0.30, 'aspect': 0.15},
}

forest_params = {
    'scene_density': 1.2,
    'simulation_years': 70,
}

species_params = {
    'oak': {
        'max_age': 110,
        'species_density': 0.016,
        'reproduction_rate': 1,
        'reproduction_radius': 5.0,
        'radius': 2.0,
        'moisture_center': 0.21,
        'moisture_width': 0.24,
        'max_slope_deg': 32,
    },
    'pine': {
        'max_age': 85,
        'species_density': 0.02,
        'reproduction_rate': 2,
        'reproduction_radius': 7.0,
        'radius': 1.8,
        'moisture_center': 0.16,
        'moisture_width': 0.26,
        'max_slope_deg': 38,
    },
}

grass_params = {
    'scene_density': 1.35,
    'patch_scale': 0.10,
    'hard_radius': 1.0,
    'falloff_radius': 3.0,
    'species_density': 0.28,
    'reproduction_rate': 3,
    'reproduction_radius': 3,
    'max_age': 8,
    'radius': 0.3,
    'simulation_years': forest_params['simulation_years'],
}

obstacle_params = {
    'specs': [
        ('boulder', 2.0, 0.45),
        ('stump', 1.2, 0.35),
        ('log', 1.4, 0.20),
    ],
    'density': 0.005,
    'min_distance': 1.8,
    'seed': 21,
}

traversability_params = {
    'resolution_factor': 3,
    'max_slope_deg': 30,
    'obstacle_influence_radius': 7.0,
    'obstacle_penalty': 0.45,
}

export_params = {
    'png_name': 'previer_height.png',
    'glb_name': 'previer_mesh.glb',
    'glb_seed': 3,
}


understory_params = {
    'scene_density': 0.2,
    'preferred_distance': 1.4,
    'avoid_radius': 0.8,
    'falloff_radius': 11.0,
    'patch_scale': 0.12,
    'patch_threshold': 0.5,
    'species_density': 0.2,
    'reproduction_rate': 2,
    'reproduction_radius': 6.0,
    'radius': 2.6,
    'max_age': 35,
    'simulation_years': forest_params['simulation_years'],
}



In [ ]:

import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Local src layout
import sys
sys.path.append('src')

from forest_gen_utils.terrain import TerrainBuilder, TerrainConfig
from forest_gen_utils.forest import ForestBuilder, ForestConfig
from forest_gen_utils.asset_dist import Species
from forest_gen_utils.asset_dist.grass import GrassDistributor
from forest_gen_utils.obstacles import ObstacleBuilder, ObstacleConfig, ObstacleSpec
from forest_gen_utils.traversability  import TraversabilityMapBuilder
from forest_gen_utils.export import ExportFactory
from forest_gen_utils.asset_dist.understory import UnderstoryDistributor

# Deterministic outputs for reproducibility
np.random.seed(7)
random.seed(7)
plt.rcParams.update({'figure.figsize': (6, 5), 'axes.titlesize': 'large'})


## 1. Build terrain
Creation of terrain is done through first defining a generator we want to use. For that we use a fluent builder. In this step we are only generating topology and variables that are derivable from it.

### Considerations:
- Combines fractal height noise, microrelief, and drainage-driven moisture from the default `TerrainBuilder`.
- Uses seed-based reproducibility and shared cube dimensions for later layers.

### Outputs:
- A terrain cube with height, drainage flow, moisture, and slope fields ready for plotting and further use.

In [ ]:
terrain_config = TerrainConfig(
    size=terrain_params['size'],
    resolution=terrain_params['resolution'],
    scale=terrain_params['scale'],
    octaves=terrain_params['octaves'],
    height_scale=terrain_params['height_scale'],
    apply_microrelief=terrain_params['apply_microrelief'],
)
terrain_generator = (
    TerrainBuilder()
    .with_noise('fractal')
    .with_microrelief(terrain_params['apply_microrelief'])
    .with_moisture_model(terrain_params['moisture_weights'])
    .build()
)
terrain = terrain_generator.generate(terrain_config)

extent = (0, terrain_config.size, 0, terrain_config.size)
terrain_summary = {
    'height': terrain.heightmap.shape,
    'flow': terrain.flow.shape,
    'slope': terrain.slope.shape,
    'moisture': terrain.moisture.shape,
}
terrain_summary


## 2. Visualize terrain layers
Here we are visualizing few of the key parameters of the generated terrain.

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

layer_titles = ['Height (m)', 'Flow accumulation', 'Moisture (0-1)', 'Slope (deg)']
layers = [
    terrain.heightmap,
    terrain.flow,
    terrain.moisture,
    np.rad2deg(terrain.slope),
]
colormaps = ['terrain', 'Blues', 'YlGnBu', 'magma']

for ax, title, layer, cmap in zip(axes.ravel(), layer_titles, layers, colormaps):
    im = ax.imshow(layer, origin='lower', cmap=cmap, extent=extent)
    ax.set_title(title)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    fig.colorbar(im, ax=ax, shrink=0.8)

# plt.show()


## 3. Configure tree species and forest generator
Considerations:
- Builds oak and pine viability from moisture and slope preferences, combining them via `CompositeViability` weights.
- Sets integer reproduction rates and annual simulations on the shared terrain cube.
Outputs:
- Species definitions, a forest generator configured with viability layers, and yearly growth results stored for plotting.

In [ ]:
extra_layers = {
   'moisture_weight': np.clip(terrain.moisture, 0.0, 1.0),
   'slope_weight': np.clip(1.0 - np.rad2deg(terrain.slope) / 45.0, 0.0, 1.0),
}

layer_weights = {\
    'moisture': 0.25,
    'slope': 0.20,
    'aspect': 0.15,
    'moisture_weight': 0.25,
    'slope_weight': 0.15,
}

def combine_layers(layers: dict[str, float]) -> float:
    weighted_values = 0.0
    total_weight = 0.0
    for name, value in layers.items():
        weight = layer_weights.get(name, 1.0)
        weighted_values += weight * value
        total_weight += weight
    return float(weighted_values / total_weight if total_weight else 1.0)

In [ ]:
def sample_layer(layer: np.ndarray, x: float, y: float) -> float:
    i = np.clip(terrain_config.transform(y), 0, layer.shape[0] - 1)
    j = np.clip(terrain_config.transform(x), 0, layer.shape[1] - 1)
    return float(layer[int(i), int(j)])

def moisture_pref(center: float, width: float):
    def wrapper(x: float, y: float) -> float:
        m = sample_layer(terrain.moisture, x, y)
        return np.exp(-((m - center) ** 2) / (2 * width ** 2))
    return wrapper

def slope_mask(max_degrees: float):
    def wrapper(x: float, y: float) -> float:
        slope_deg = sample_layer(terrain.slope, x, y)  # already degrees
        return float(np.clip(1.0 - slope_deg / max_degrees, 0.0, 1.0))
    return wrapper

oak_params = species_params['oak']
pine_params = species_params['pine']
oak_m = moisture_pref(oak_params["moisture_center"], oak_params["moisture_width"])
oak_s = slope_mask(oak_params["max_slope_deg"])

pine_m = moisture_pref(pine_params["moisture_center"], pine_params["moisture_width"])
pine_s = slope_mask(pine_params["max_slope_deg"])
oak = Species(
    name="Oak",
    max_age=oak_params["max_age"],
    species_density=oak_params["species_density"],
    reproduction_rate=oak_params["reproduction_rate"],
    reproduction_radius=oak_params["reproduction_radius"],
    radius=oak_params["radius"],
    viability_map=lambda x, y, _m=oak_m, _s=oak_s: _m(x, y) * _s(x, y),
)

pine = Species(
    name="Pine",
    max_age=pine_params["max_age"],
    species_density=pine_params["species_density"],
    reproduction_rate=pine_params["reproduction_rate"],
    reproduction_radius=pine_params["reproduction_radius"],
    radius=pine_params["radius"],
    viability_map=lambda x, y, _m=pine_m, _s=pine_s: _m(x, y) * _s(x, y),
)

forest = (
    ForestBuilder()
    .with_size((terrain_config.size, terrain_config.size))
    .with_terrain(terrain)
    .with_terrain_viability_layers(extra_layers, combine=combine_layers)
    .add_species('tree', oak)
    .add_species('tree', pine)
    .build()
)


import numpy as np

def viability_report(sp, size=50.0, n=5000):
    rng = np.random.default_rng(0)
    pts = rng.uniform([0,0],[size,size], size=(n,2))
    vals = np.array([sp.viability_map(x,y) for x,y in pts], dtype=float)
    vals = np.clip(vals, 0.0, 1.0)
    print(sp.name, "mean", vals.mean(), "p50", np.quantile(vals,0.5), "p90", np.quantile(vals,0.9), "max", vals.max())

viability_report(pine)
viability_report(oak)

m = terrain.moisture
s = terrain.slope

print("moisture median", float(np.median(m)), "p10", float(np.quantile(m,0.1)), "p90", float(np.quantile(m,0.9)))
print("slope median", float(np.median(s)), "p90", float(np.quantile(s,0.9)), "max", float(np.max(s)))




forest_states = []
state = forest.generate(ForestConfig(scene_density=forest_params['scene_density'], years=0))
for _ in range(forest_params['simulation_years']):
    state.run_state(1)
    forest_states.append(deepcopy(state))

len(forest_states[-1])


## 4. Visualize forest growth
Considerations:
- Overlays yearly snapshots to show how oak and pine spread relative to moisture/slope suitability.
Outputs:
- Side-by-side plots of occupancy per year for each species to validate the simulation dynamics.

In [ ]:

species_colors = {'Oak': '#8B5A2B', 'Pine': '#228B22'}

fig, axes = plt.subplots(1, len(forest_states), figsize=(4 * len(forest_states), 4), sharex=True, sharey=True)
if len(forest_states) == 1:
    axes = [axes]

for idx, (ax, snapshot) in enumerate(zip(axes, forest_states), start=1):
    xs, ys, colors = [], [], []
    for plant in snapshot:
        xs.append(plant.coords[0])
        ys.append(plant.coords[1])
        colors.append(species_colors.get(plant.species.name, '#555555'))
    ax.scatter(xs, ys, s=12, c=colors, alpha=0.8)
    ax.set_title(f'Year {idx}')
    ax.set_xlim(0, terrain_config.size)
    ax.set_ylim(0, terrain_config.size)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

# plt.show()


## 5. Add grass layer
Considerations:
- Uses `GrassBuilder` preferences for lower slopes and open spaces, masking out tree locations.
Outputs:
- A grass density map and corresponding visualization highlighting grassy regions between trees.

In [ ]:

tree_positions = [plant.coords for plant in forest_states[-1]]
grass_generator = GrassDistributor(
    terrain,
    tree_positions,
    patch_scale=grass_params['patch_scale'],
    hard_radius=grass_params['hard_radius'],
    falloff_radius=grass_params['falloff_radius'],
    max_age=grass_params['max_age'],
    species_density=grass_params['species_density'],
    reproduction_rate=grass_params['reproduction_rate'],
    reproduction_radius=grass_params['reproduction_radius'],
    radius=grass_params['radius'],
)
grass_state = grass_generator.generate(
    ForestConfig(scene_density=grass_params['scene_density'], years=0)
)
print("grass founders:", len(list(grass_state)))
grass_states = []
for _ in range(grass_params['simulation_years']):
    grass_state.run_state(1)
    print("grass step:", len(list(grass_state)))
    grass_states.append(deepcopy(grass_state))

final_grass_state = grass_states[-1] if grass_states else grass_state

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(terrain.heightmap, cmap='Greys', origin='lower', extent=extent, alpha=0.35)
ax.scatter(*zip(*tree_positions), s=8, c='#2d6a4f', label='Trees', alpha=0.8)
ax.scatter(
    [p.coords[0] for p in final_grass_state],
    [p.coords[1] for p in final_grass_state],
    s=6,
    c='#8fd694',
    label='Grass',
    alpha=0.6,
)
ax.legend(loc='upper right')
ax.set_title('Grass distribution around forest canopy after yearly spread')
ax.set_xlim(0, terrain_config.size)
ax.set_ylim(0, terrain_config.size)
# plt.show()

## 6. Add understory layer
Considerations:
- Uses `UnderstoryDistributor` to favor shaded patches around canopy trees while avoiding trunks.
- Reuses terrain viability layers so shrub placement respects slope and moisture limits.
Outputs:
- An understory distribution plot showing shrubs clustering beneath the canopy.

In [ ]:
understory_generator = UnderstoryDistributor(
    terrain,
    tree_positions,
    preferred_distance=understory_params['preferred_distance'],
    avoid_radius=understory_params['avoid_radius'],
    falloff_radius=understory_params['falloff_radius'],
    patch_scale=understory_params['patch_scale'],
    patch_threshold=understory_params['patch_threshold'],
    species_density=understory_params['species_density'],
    reproduction_rate=understory_params['reproduction_rate'],
    reproduction_radius=understory_params['reproduction_radius'],
    radius=understory_params['radius'],
    max_age=understory_params['max_age'],
)
understory_state = understory_generator.generate(
    ForestConfig(scene_density=understory_params['scene_density'], years=0)
)
print("understory founders:", len(list(understory_state)))
understory_states = []
for _ in range(understory_params['simulation_years']):
    understory_state.run_state(1)
    understory_states.append(deepcopy(understory_state))
    if(_ % 5 == 0):
        print("understory step:", len(list(understory_state)))


final_understory_state = understory_states[-1] if understory_states else understory_state

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(terrain.heightmap, cmap='Greys', origin='lower', extent=extent, alpha=0.35)
ax.scatter(*zip(*tree_positions), s=8, c='#2d6a4f', label='Trees', alpha=0.8)
if final_understory_state:
    ax.scatter(
        [p.coords[0] for p in final_understory_state],
        [p.coords[1] for p in final_understory_state],
        s=8,
        c='#c3a995',
        label='Understory',
        alpha=0.7,
    )
ax.legend(loc='upper right')
ax.set_title('Understory shrubs beneath canopy after yearly spread')
ax.set_xlim(0, terrain_config.size)
ax.set_ylim(0, terrain_config.size)
# plt.show()
if final_understory_state:
    coords = np.array([p.coords for p in final_understory_state], dtype=float)
    dist_matrix = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    dist_matrix[dist_matrix == 0] = np.nan
    mean_spacing = float(np.nanmean(np.nanmin(dist_matrix, axis=1))) if np.isfinite(dist_matrix).any() else float("nan")
    print(f"Mean nearest-neighbor distance: {mean_spacing:.2f} m")
    if mean_spacing < 3.0:
        print('WARNING: Shrub spacing below 3 m target')
else:
    print('No understory shrubs generated.')



## 7. Sample obstacles
Considerations:
- Samples obstacle types with per-type radius ranges, global minimum spacing, and reproducible seeds via `ObstacleGenerator`.
Outputs:
- A list of obstacle positions/radii and a plot showing where rocks, stumps, or other blockers land on the terrain.

In [ ]:
obstacle_builder = (
    ObstacleBuilder()
    .with_specs(tuple(ObstacleSpec(name, radius=radius, weight=weight) for name, radius, weight in obstacle_params['specs']))
    .with_seed(obstacle_params['seed'])
)
obstacle_generator = obstacle_builder.build()
obstacle_config = ObstacleConfig(size=(terrain_config.size, terrain_config.size), density=obstacle_params['density'], min_distance=obstacle_params['min_distance'])
obstacles = obstacle_generator.generate(obstacle_config)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(terrain.heightmap, cmap='bone', origin='lower', extent=extent, alpha=0.3)
for obstacle in obstacles:
    circle = plt.Circle(obstacle.coords, obstacle.radius, color='#c77dff', alpha=0.5)
    ax.add_patch(circle)
    ax.plot(*obstacle.coords, 'o', color='#560bad', ms=4)
ax.set_title(f"{len(obstacles)} obstacles with spacing buffer")
ax.set_xlim(0, terrain_config.size)
ax.set_ylim(0, terrain_config.size)
# plt.show()


## 8. Traversability map
Considerations:
- Blends slope penalties with obstacle influence to approximate navigation cost.
Outputs:
- A traversability heatmap that grades easy vs. hard routes across the terrain.

In [ ]:
traversability = TraversabilityMapBuilder(terrain, resolution_factor=traversability_params['resolution_factor'], max_slope_deg=traversability_params['max_slope_deg'])
traversability.add_obstacle_score([obs.coords for obs in obstacles], obstacle_influence_radius=traversability_params['obstacle_influence_radius'], obstacle_penalty=traversability_params['obstacle_penalty'])
score = traversability.get_score()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(score, origin='lower', cmap='viridis', extent=extent)
ax.set_title('Traversability (1=easy, 0=blocked)')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
fig.colorbar(im, ax=ax, shrink=0.8)
# plt.show()


## 9. Export terrain assets
Considerations:
- Converts the final height/vegetation layers into PNGs and GLB meshes through the `ExportFactory`.
Outputs:
- Serialized assets in the `outputs/` directory for downstream tools and viewers.

In [ ]:
tree_count = len(forest_states[-1])
grass_count = len(final_grass_state)
understory_count = len(final_understory_state) if final_understory_state is not None else 0
obstacle_count = len(obstacles)

print(f"Final entity counts -> Trees: {tree_count}, Grass: {grass_count}, Understory: {understory_count}, Obstacles: {obstacle_count}")


In [ ]:
export_dir = Path('outputs')
export_dir.mkdir(exist_ok=True)

png_exporter = ExportFactory.create('png', max_elevation=np.max(terrain.heightmap))
png_path = export_dir / export_params['png_name']
png_exporter.export(terrain.heightmap, str(png_path))

glb_exporter = ExportFactory.create('glb', resolution=terrain_config.resolution, max_elevation=np.max(terrain.heightmap), seed=export_params['glb_seed']
)
glb_path = export_dir / export_params['glb_name']
glb_exporter.export(terrain.heightmap, str(glb_path))

png_path, glb_path
